# Baraka Protocol — Simulation Explorer

Interactive Jupyter notebook for exploring all 4 simulation modules.

**Prerequisite:**
```bash
cd simulations
pip install -r requirements.txt
jupyter notebook notebooks/baraka_simulation_explorer.ipynb
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

print('Baraka Protocol Simulation Explorer — Ready')

## Module 1: cadCAD Monte Carlo

In [ ]:
from cadcad.run import run_simulation, plot_results

# Run 10 Monte Carlo simulations of 30 days each
df = run_simulation(steps=720, n_runs=10)
print(f'Generated {len(df):,} data points across {df.run.nunique()} runs')
df.head()

In [ ]:
# Plot funding rate distribution — verify ι=0 (no systematic positive bias)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['funding_rate'] * 10000, bins=80, edgecolor='black', color='steelblue', alpha=0.7)
axes[0].axvline(0, color='red', linewidth=2, label='Zero (ι=0 target)')
axes[0].set_title('Funding Rate Distribution (all runs, all steps)')
axes[0].set_xlabel('bps')
axes[0].legend()

mean_fr = df.groupby('step')['funding_rate'].mean()
axes[1].plot(mean_fr.index, mean_fr * 10000, color='purple', linewidth=1.5)
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Mean Funding Rate Over Time')
axes[1].set_xlabel('Interval (1hr)')
axes[1].set_ylabel('bps')

plt.tight_layout()
plt.show()

print(f'Mean funding rate: {df["funding_rate"].mean() * 10000:.3f} bps (should be ~0 for ι=0)')

In [ ]:
# Insurance fund health
last_step = df[df['step'] == df['step'].max()]
print('Insurance Fund at end of simulation:')
print(f'  Mean:     ${last_step["insurance_fund_balance"].mean():,.2f}')
print(f'  Min:      ${last_step["insurance_fund_balance"].min():,.2f}')
print(f'  Insolvent runs: {last_step["protocol_insolvent"].sum()}/{len(last_step)}')

## Module 2: Reinforcement Learning Trader

In [ ]:
from agents.rl_trader import BarakaTraderEnv

# Run 5 random-action episodes to verify environment works
pnls = []
for ep in range(5):
    env  = BarakaTraderEnv(seed=ep)
    obs, _ = env.reset()
    done = False
    while not done:
        action = env.action_space.sample()
        obs, r, done, _, info = env.step(action)
    pnls.append(info['total_pnl'])
    print(f'Episode {ep+1}: PnL=${info["total_pnl"]:,.2f}, Collateral=${info["free_collateral"]:,.2f}')

print(f'\nMean PnL (random policy): ${np.mean(pnls):,.2f}')

## Module 3: Game Theory — ι=0 Nash Equilibrium

In [ ]:
from game_theory.funding_game import compare_iota_regimes, build_payoff_matrices

# Compare ι=0 vs ι=0.005 — riba test
results = compare_iota_regimes(n_simulations=50, steps=100,
                               iota_values=[0.0, 0.001, 0.002, 0.005])

fig, ax = plt.subplots(figsize=(9, 4))
iotas = list(results.keys())
net   = [results[i]['net_transfer'] for i in iotas]
colors = ['green' if not results[i]['is_riba'] else 'red' for i in iotas]

ax.bar([f'ι={i:.3f}' for i in iotas], net, color=colors)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Net Transfer from Longs to Protocol (>0 = riba)')
ax.set_ylabel('USD (per $100k OI)')
plt.tight_layout()
plt.show()

print('\nConclusion:')
for iota, r in results.items():
    status = 'RIBA ✗' if r['is_riba'] else 'Shariah-compliant ✓'
    print(f'  ι={iota:.3f}: net_transfer=${r["net_transfer"]:,.2f} → {status}')

## Module 4: Stress Test Scenarios

In [ ]:
from scenarios.flash_crash import run_scenario, print_scenario_summary

# Run flash crash
df_crash = run_scenario('flash_crash', n_positions=50)
print_scenario_summary(df_crash, 'flash_crash')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df_crash['step'], df_crash['mark_price'], label='Mark')
axes[0].plot(df_crash['step'], df_crash['index_price'], label='Index', linestyle='--')
axes[0].set_title('Flash Crash: Price')
axes[0].legend()

axes[1].plot(df_crash['step'], df_crash['insurance_fund'], color='green')
axes[1].axhline(0, color='red', linestyle='--', label='Insolvency')
axes[1].set_title('Insurance Fund During Flash Crash')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Run all 5 scenarios and compare
from scenarios.flash_crash import SCENARIOS

summary = []
for name in SCENARIOS:
    df_s = run_scenario(name, n_positions=40)
    summary.append({
        'scenario': name,
        'survived': not df_s['insolvent'].any(),
        'min_insurance': df_s['insurance_fund'].min(),
        'total_liquidations': df_s['liquidations'].sum(),
        'iota_ok': not df_s['iota_violated'].any(),
    })

pd.DataFrame(summary)